In [1]:
import os
import re
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
client = OpenAI()


# -----------------------------
# Hilfsfunktionen
# -----------------------------

OUTPUT_CONTRACT_DEFAULT = """
Return ONLY valid JSON (no markdown, no commentary).

Schema:
{
  "A": {"severity": <int>},
  "B": {"severity": <int>},
  "C": {"severity": <int>},
  "D": {"severity": <int>},
  "E": {"severity": <int>}
}
""".strip()

DOMAINS = ["A", "B", "C", "D", "E"]

def next_free_run_path(results_dir: Path, model: str, temperature: float) -> Path:
    """
    Erzeugt:
      run_7_gpt-5.2_t1.csv
    mit automatisch nächster Run-Nummer.
    """

    results_dir.mkdir(parents=True, exist_ok=True)

    # vorhandene run_X_*.csv suchen
    nums = []
    for p in results_dir.glob("run_*.csv"):
        m = re.search(r"run_(\d+)_", p.name)
        if m:
            nums.append(int(m.group(1)))

    n = (max(nums) + 1) if nums else 1

    # Model-Name dateisystem-sicher machen
    safe_model = re.sub(r"[^\w\.-]+", "-", model)

    # Temperature hübsch formatieren
    if float(temperature).is_integer():
        temp_str = f"t{int(temperature)}"
    else:
        temp_str = f"t{str(temperature).replace('.', '_')}"

    filename = f"run_{n}_{safe_model}_{temp_str}.csv"
    return results_dir / filename


def _extract_json_from_response_obj(resp_body: dict) -> dict:
    out = resp_body.get("output", [])
    texts = []
    for item in out:
        for c in item.get("content", []):
            if c.get("type") in ("output_text", "text") and "text" in c:
                texts.append(c["text"])
    if not texts and resp_body.get("output_text"):
        texts = [resp_body["output_text"]]
    if not texts:
        raise ValueError("Kein output_text gefunden")

    text = "\n".join(texts).strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        raise ValueError(f"Kein JSON im Text: {text[:200]}...")
    return json.loads(m.group(0))


def parse_batch_output_jsonl_text_to_df(jsonl_text: str) -> pd.DataFrame:
    by_call = {}

    for line in jsonl_text.splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)

        call_id = rec.get("custom_id")  # jetzt: "Notruf_12" (ohne __A)
        if not call_id:
            continue

        by_call.setdefault(call_id, {"file": call_id})

        err = rec.get("error")
        if err:
            # Falls Request komplett fehlgeschlagen ist: alles None setzen
            for d in DOMAINS:
                by_call[call_id][f"severity_{d}"] = None
                #by_call[call_id][f"confidence_{d}"] = None
                #by_call[call_id][f"findings_{d}"] = json.dumps([], ensure_ascii=False)
                #by_call[call_id][f"error_{d}"] = json.dumps(err, ensure_ascii=False)
            continue

        body = (rec.get("response") or {}).get("body") or {}
        try:
            data = _extract_json_from_response_obj(body)  # erwartet {"A": {...}, ...}
        except Exception as e:
            for d in DOMAINS:
                by_call[call_id][f"severity_{d}"] = None
                #by_call[call_id][f"confidence_{d}"] = None
                #by_call[call_id][f"findings_{d}"] = json.dumps([], ensure_ascii=False)
                #by_call[call_id][f"error_{d}"] = f"parse_error: {e}"
            continue

        # Domains aus dem JSON ziehen
        for d in DOMAINS:
            block = data.get(d, None)
            if not isinstance(block, dict):
                by_call[call_id][f"severity_{d}"] = None
                #by_call[call_id][f"confidence_{d}"] = None
                #by_call[call_id][f"findings_{d}"] = json.dumps([], ensure_ascii=False)
                #by_call[call_id][f"error_{d}"] = f"missing_or_invalid_block_{d}"
                continue

            sev = block.get("severity", None)
            #conf = block.get("confidence", None)
           # findings = block.get("findings", [])
            #if findings is None:
            #    findings = []
            #if not isinstance(findings, list):
            #    findings = [str(findings)]

            by_call[call_id][f"severity_{d}"] = sev
            #by_call[call_id][f"confidence_{d}"] = conf
            #by_call[call_id][f"findings_{d}"] = json.dumps(findings, ensure_ascii=False)
            by_call[call_id][f"error_{d}"] = None

    df = pd.DataFrame(list(by_call.values()))
    df["call_nr"] = df["file"].str.extract(r"(\d+)").astype("Int64")
    df = df.sort_values(["call_nr", "file"], na_position="last").reset_index(drop=True)

    cols = ["file", "call_nr"]
    for d in DOMAINS:
        cols += [f"severity_{d}"]
    for c in cols:
        if c not in df.columns:
            df[c] = None
    return df[cols]


def _download_file_text(client: OpenAI, file_id: str) -> str:
    """
    Robust: lädt die Datei-Inhalte als Text.
    (SDK-Details können variieren; das deckt die üblichen Fälle ab.)
    """
    resp = client.files.content(file_id)
    # je nach SDK: resp kann bytes, ein HTTP Response-Objekt mit .read(), oder schon text sein
    if hasattr(resp, "read"):
        data = resp.read()
    else:
        data = resp
    if isinstance(data, bytes):
        return data.decode("utf-8", errors="replace")
    return str(data)


# -----------------------------
# Hauptfunktion
# -----------------------------
def run_abcde_batch_experiment(
    model: str,
    temperature: float,
    *,
    calls_dir: Path,
    guidelines_dir: Path,   # enthält jetzt GENAU EINE Guideline-Datei (siehe unten)
    results_dir: Path,
    completion_window: str = "24h",
    output_contract: str = OUTPUT_CONTRACT_DEFAULT,  # muss A..E Schema enthalten!
    wait: bool = True,
    poll_every_seconds: int = 20,
    max_wait_seconds: int = 60 * 60,
):
    """
    Führt ein komplettes ABCDE-Batch-Experiment aus und speichert CSV als nächste freie run_X.csv.

    NEU:
    - Pro Notruf genau 1 Request (custom_id = Notruf_12)
    - Guideline ist nur 1 Text (Master Guideline)
    - Output muss JSON mit Schlüsseln A,B,C,D,E enthalten (jeweils severity/confidence/findings)
    """

    # OpenAI Client
    load_dotenv()
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise EnvironmentError("OPENAI_API_KEY fehlt (env).")
    client = OpenAI()

    calls_dir = Path(calls_dir).resolve()
    guidelines_dir = Path(guidelines_dir).resolve()
    results_dir = Path(results_dir).resolve()

    assert calls_dir.exists(), f"calls_dir nicht gefunden: {calls_dir}"
    assert guidelines_dir.exists(), f"guidelines_dir nicht gefunden: {guidelines_dir}"

    call_files = sorted(list(calls_dir.glob("*.md")) + list(calls_dir.glob("*.txt")))
    assert call_files, f"Keine Notrufe gefunden in {calls_dir}"

    # -----------------------------
    # Master-Guideline laden (GENAU 1 Datei)
    # -----------------------------
    prompt = ("You are an expert in analysing emergency calls. \n"
              "You should evaluate the call transcript according the ABCDE schema. \n"
              "ABCDE Schema: Airway, Breathing, Circulation, Disability, Exposure.\n"
              "Give every category a severity score from 0 to 3.\n"
              "**0** – No information available; no conclusions possible\n"  
            "**1** – No or only minor impairment suspected\n"  
            "**2** – Significant impairment present; no acute life threat suspected (ambulance indicated)\n"  
            "**3** – Severe impairment; vital threat possible or likely; immediate intervention required (urgent ambulance, possibly physician response unit)")

    # Batch input schreiben (immer in results_dir ablegen)
    results_dir.mkdir(parents=True, exist_ok=True)
    batch_input_path = results_dir / "batch_input_abcde.jsonl"

    with batch_input_path.open("w", encoding="utf-8") as f:
        for fp in call_files:
            transcript = fp.read_text(encoding="utf-8").strip()

            req_body = {
                "model": model,
                # Achtung: manche Modelle (z.B. gpt-5-mini) unterstützen temperature nicht
                # Wenn du temperature nur setzen willst, wenn != None und erlaubt, dann ggf. dynamisch ergänzen.
                # "temperature": temperature,
                "input": [
                    {"role": "system", "content": [{"type": "input_text", "text": prompt}]},
                    {"role": "system", "content": [{"type": "input_text", "text": output_contract}]},
                    {"role": "user",   "content": [{"type": "input_text", "text": transcript}]},
                ],
                "text": {"format": {"type": "json_object"}},
            }

            # OPTIONAL: wenn du temperature nur bei bestimmten Modellen setzen willst,
            # hier “safe add” (auskommentieren wenn nicht benötigt):
            # if temperature is not None and not ("gpt-5-mini" in model):
            #     req_body["temperature"] = float(temperature)

            req = {
                "custom_id": f"{fp.stem}",   # <-- nur ein Request pro Notruf!
                "method": "POST",
                "url": "/v1/responses",
                "body": req_body,
            }
            f.write(json.dumps(req, ensure_ascii=False) + "\n")

    # Upload + Batch create
    upload = client.files.create(file=batch_input_path.open("rb"), purpose="batch")
    batch = client.batches.create(
        input_file_id=upload.id,
        endpoint="/v1/responses",
        completion_window=completion_window,
    )

    # Wenn nicht warten: sofort zurückgeben
    if not wait:
        return {
            "status": batch.status,
            "batch_id": batch.id,
            "batch_input_path": str(batch_input_path),
            "output_jsonl_path": None,
            "csv_path": None,
            "df": None,
        }

    # Polling
    t0 = time.time()
    while True:
        b = client.batches.retrieve(batch.id)
        if b.status in ("completed", "failed", "cancelled", "expired"):
            break
        if max_wait_seconds is not None and (time.time() - t0) > max_wait_seconds:
            return {
                "status": b.status,
                "batch_id": batch.id,
                "batch_input_path": str(batch_input_path),
                "output_jsonl_path": None,
                "csv_path": None,
                "df": None,
                "note": (
                    f"Nicht fertig nach max_wait_seconds={max_wait_seconds}. "
                    f"Setze wait=False oder erhöhe max_wait_seconds."
                ),
            }
        time.sleep(poll_every_seconds)

    if b.status != "completed":
        err_text = None
        if getattr(b, "error_file_id", None):
            err_text = _download_file_text(client, b.error_file_id)
        return {
            "status": b.status,
            "batch_id": batch.id,
            "batch_input_path": str(batch_input_path),
            "output_jsonl_path": None,
            "csv_path": None,
            "df": None,
            "error_file_text": err_text,
        }

    # Output JSONL downloaden und lokal speichern
    if not b.output_file_id:
        raise RuntimeError("Batch completed, aber output_file_id fehlt.")
    output_text = _download_file_text(client, b.output_file_id)

    output_jsonl_path = results_dir / f"batch_output_{batch.id}.jsonl"
    output_jsonl_path.write_text(output_text, encoding="utf-8")

    # JSONL -> DF -> CSV
    # WICHTIG: parse_batch_output_jsonl_text_to_df muss auf 1-request-pro-call angepasst sein
    # (custom_id ohne __A und JSON enthält A..E)
    df = parse_batch_output_jsonl_text_to_df(output_text)

    csv_path = next_free_run_path(results_dir, model=model, temperature=temperature)
    df.to_csv(csv_path, index=False, encoding="utf-8")

    return {
        "status": b.status,
        "batch_id": batch.id,
        "batch_input_path": str(batch_input_path),
        "output_jsonl_path": str(output_jsonl_path),
        "csv_path": str(csv_path),
        "df": df,
    }

In [2]:
from pathlib import Path

res = run_abcde_batch_experiment(
    model="gpt-5.2",
    temperature=1,
    calls_dir=Path("../../emergency_calls/calls_german"),
    guidelines_dir=Path("guidelines"),
    results_dir=Path("results"),
    completion_window="24h",
    wait=True,                 # setzt auf False, wenn du nicht pollen willst
    max_wait_seconds=None,     # 1h; bei 24h window ggf. erhöhen oder wait=False
)

print(res["status"], res["csv_path"])
df = res["df"]
df.head()

completed /Users/s.franke/Development/master_clean/experiments/experiment_0/results/run_1_gpt-5.2_t1.csv


,file,call_nr,severity_A,severity_B,severity_C,severity_D,severity_E
0,Notruf_1,1,1,2,3,1,0
1,Notruf_2,2,1,1,2,3,0
2,Notruf_3,3,1,1,1,2,1
3,Notruf_4,4,1,2,1,1,1
4,Notruf_5,5,1,2,1,1,2


In [3]:
res = run_abcde_batch_experiment(
    model="gpt-5.2",
    temperature=0.5,
    calls_dir=Path("../../emergency_calls/calls_german"),
    guidelines_dir=Path("guidelines"),
    results_dir=Path("results"),
    completion_window="24h",
    wait=True,                 # setzt auf False, wenn du nicht pollen willst
    max_wait_seconds=None,     # 1h; bei 24h window ggf. erhöhen oder wait=False
)

print(res["status"], res["csv_path"])
df = res["df"]
df.head()

completed /Users/s.franke/Development/master_clean/experiments/experiment_0/results/run_2_gpt-5.2_t0_5.csv


,file,call_nr,severity_A,severity_B,severity_C,severity_D,severity_E
0,Notruf_1,1,1,2,3,1,0
1,Notruf_2,2,1,1,2,3,0
2,Notruf_3,3,1,1,0,2,1
3,Notruf_4,4,1,2,1,1,1
4,Notruf_5,5,1,2,1,1,2


In [4]:
res = run_abcde_batch_experiment(
    model="gpt-5.2",
    temperature=0,
    calls_dir=Path("../../emergency_calls/calls_german"),
    guidelines_dir=Path("guidelines"),
    results_dir=Path("results"),
    completion_window="24h",
    wait=True,                 # setzt auf False, wenn du nicht pollen willst
    max_wait_seconds=None,     # 1h; bei 24h window ggf. erhöhen oder wait=False
)

print(res["status"], res["csv_path"])
df = res["df"]
df.head()

completed /Users/s.franke/Development/master_clean/experiments/experiment_0/results/run_3_gpt-5.2_t0.csv


,file,call_nr,severity_A,severity_B,severity_C,severity_D,severity_E
0,Notruf_1,1,1,2,3,1,0
1,Notruf_2,2,1,1,2,3,0
2,Notruf_3,3,1,1,1,2,2
3,Notruf_4,4,1,2,1,1,1
4,Notruf_5,5,1,1,1,1,2
